# 21. Embedding tries sweep

## 실험 목적

본 실험(notebook 05)에서 MS는 6x6 이상에서 embedding에 실패했습니다. 원인 후보가 셋인데 관측만으로는 구분되지 않습니다.

| 관측 | 원인 |
|---|---|
| timeout/tries를 늘렸더니 성공 | **(a) search budget** |
| Pegasus 실패, Zephyr 성공 | **(b) topology / connectivity** |
| 충분한 budget + Zephyr에서도 실패 | **(c) MS formulation 자체** |

embedding 탐색은 확률적이므로 seed 하나로 판정하지 않고 여러 seed의 **성공률**로 봅니다.

embedding 가능 여부는 QUBO의 **그래프 구조에만** 의존하고 penalty 값과는 무관합니다. 따라서 본 실험의 기본 설정으로 QUBO를 한 번만 만들어 씁니다.

이 notebook은 **timeout을 고정하고 tries를 훑습니다.**

`tries`는 minorminer가 서로 다른 시작점에서 재시도하는 횟수입니다. timeout이 전체 시간 예산이라면 tries는 **탐색의 다양성**에 해당합니다. 둘은 서로 다른 종류의 budget이므로 나누어 봅니다.

고정할 timeout은 notebook 14의 결과를 보고 정하십시오. 성공률이 포화되기 시작하는 값이 적절합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
STUDY = config["embedding_study"]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


In [ ]:
QA_DRYRUN = False

TARGETS = [
    ("4x4", "MS"),
    ("6x6", "MS"),
    ("8x8", "MS"),
    ("15x15", "MS"),
]

TOPOLOGIES = [tuple(item) for item in STUDY["topologies"]]
TRIES_GRID = list(STUDY["tries_grid"])
SEEDS = list(STUDY["seeds"])
FIXED_TIMEOUT = int(STUDY["tries_sweep_timeout"])

print(f"모드          : {'이상적 topology 그래프' if QA_DRYRUN else '실제 QPU working graph'}")
print(f"tries grid    : {TRIES_GRID}")
print(f"timeout(고정) : {FIXED_TIMEOUT}s")
print(f"seed          : {SEEDS}")

## 실행 시간 추정

tries를 늘리면 minorminer가 재시도를 더 많이 하지만, **timeout이 전체 상한**이므로 한 번의 실행이 `FIXED_TIMEOUT`을 넘지는 않습니다. 따라서 최악의 경우는 아래와 같습니다.

In [ ]:
from src import embedding_study as ES

budget = ES.estimate_budget(
    TARGETS, TOPOLOGIES, [FIXED_TIMEOUT] * len(TRIES_GRID), SEEDS
)
print(f"총 실행 횟수      : {budget['runs']:.0f}")
print(f"최악의 경우(전부 실패): {budget['worst_case_minutes']:.0f}분 "
      f"({budget['worst_case_hours']:.1f}시간)")

## tries sweep 실행

In [ ]:
results = ES.run_tries_sweep(
    targets=TARGETS,
    config=config,
    data_dir=DATA_DIR,
    tries_grid=TRIES_GRID,
    seeds=SEEDS,
    timeout=FIXED_TIMEOUT,
    topologies=TOPOLOGIES,
    qa_dryrun=QA_DRYRUN,
)
print(f"\n{len(results)} 회 실행 완료")

In [ ]:
from src.persistence import save_table

OUTPUT = PROCESSED_DIR / "embedding_study.csv"
if OUTPUT.exists():
    previous = pd.read_csv(OUTPUT)
    key = ["instance", "formulation", "topology", "sweep",
           "timeout", "tries", "seed"]
    merged = pd.concat([previous, results], ignore_index=True)
    merged = merged.drop_duplicates(subset=key, keep="last")
else:
    merged = results
save_table(merged, PROCESSED_DIR, "embedding_study.csv")
print(f"저장: {OUTPUT} ({len(merged)} 행)")
print(merged.groupby(["sweep", "topology"])["success"].agg(["count", "mean"]))

In [ ]:
pivot = results.pivot_table(
    index=["instance", "logical_variables"],
    columns=["topology", "tries"],
    values="success",
    aggfunc="mean",
)
(pivot * 100).round(0)